# Geneformer V2-104M — DIMER E2E single-cell state classification tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/geneformer-single-cell-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/geneformer-single-cell-pipeline/blob/main/tutorials/geneformer_single_cell_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-ctheodoris%2FGeneformer-ffcc4d?style=flat)](https://huggingface.co/ctheodoris/Geneformer) [![Nature 2023](https://img.shields.io/badge/Nature-Transfer%20learning%20in%20network%20biology-b31b1b.svg)](https://www.nature.com/articles/s41586-023-06139-9) [![bioRxiv 2024](https://img.shields.io/badge/bioRxiv-2024.08.16.608180-b31b1b.svg)](https://doi.org/10.1101/2024.08.16.608180)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** rank-value encoding of single-cell transcriptomes, cell embeddings, and bounded cell-state classification fine-tuning

**This notebook is standalone.** It carries the repository's package (3 modules under `src/geneformer_single_cell_pipeline/`, at revision `uncommitted`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `1f7fbae4e469a5f4f1af8c111a529cfe1b3829f5` (~421 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned Geneformer V2-104M snapshot (6 files, ~421 MB: the checkpoint plus the three gene dictionaries), loads those dictionaries through a restricted unpickler, generates a deterministic 64-cell tutorial dataset over real human Ensembl gene ids (no download), validates it against the cell contract, splits it into stratified train/validation/test sets, rank-value encodes cells and inspects one encoding, computes cell embeddings, measures a majority-class and a library-size baseline on the test split, runs a bounded AdamW fine-tuning of the cell-state classification head and the last two encoder layers, evaluates accuracy, macro-F1 and AUROC on the held-out test split, classifies six freshly generated cells, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify prediction parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On a CPU runtime the whole path takes a few minutes, most of it in the fine-tuning cell.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own labelled cells as a genes-as-columns CSV (`id`, one column per gene, `label`), a JSON array or a JSONL file of `{id, counts, label}` records. It passes through the same validation, stratified split, baselines, rank-value encoding, adaptation, held-out evaluation, inference, artifact export and reload-parity cells as the synthetic sample. The expected schema, the identifier requirements and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

Geneformer is a transformer pretrained on single-cell transcriptomes, and its central idea is the input format rather than the architecture. A cell is not fed to the model as a vector of counts: each gene's expression is divided by that gene's median expression across the pretraining corpus, the genes are **ranked** by the result, and the ranked gene identifiers become the token sequence. Housekeeping genes that are high in every cell are scaled down; a transcription factor that is lowly expressed but rarely expressed at all moves up. The consequence is that sequencing depth and absolute count scale largely drop out, and what the model reads is the order of genes within the cell. The pinned V2-104M checkpoint is a 12-layer BERT encoder with a 20,275-token gene vocabulary and a 4,096-token input, pretrained on ~104 million human transcriptomes.

This tutorial implements that encoding in the notebook, from the pinned gene dictionaries, and then adapts the encoder to a cell-state classification task. The tutorial dataset is synthetic but uses real human Ensembl ids, and it is built so that library size carries no signal: every cell is scaled to the same total counts and detects the same genes, and the two classes differ only in which of two median-matched gene programmes ranks above the other. A baseline that thresholds total counts therefore cannot separate them — which is exactly the failure mode rank-value encoding exists to avoid.

**Learning objectives:** install the pinned runtime; inspect the carried pipeline, dataset and metrics modules; stage and digest-verify the Geneformer snapshot and load its gene dictionaries through a restricted unpickler; validate a labelled cell dataset against explicit ceilings and split it without leakage; read a rank-value encoding and see what it drops or truncates; extract cell embeddings; measure majority-class and library-size baselines; run a bounded fine-tuning with explicit hyperparameters and a recorded trainable-parameter set; evaluate accuracy, macro-F1 and AUROC on an independent test split; classify new cells; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** in-silico perturbation, gene classification, multitask or continual learning, masked-gene prediction, the V1-10M / V2-316M / cancer-tuned checkpoints, and reading `.h5ad`, `.loom` or other single-cell file formats. The repository exposes none of these; its input is a mapping of gene identifier to count.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). CPU is enough — the default fine-tuning takes a few minutes — and CUDA is used automatically when present.
- **Knowledge:** what a single-cell count matrix is, why library size differs between cells, and how accuracy, macro-F1 and AUROC differ.
- **Data contract:** records are `{{id, counts, label}}` where `counts` maps a human Ensembl gene id (`ENSG...`) or an approved gene symbol to a non-negative count. At least 10 genes must be detected per cell and carry both a Geneformer token and a corpus median; at most 4,094 ranked genes fit the input; at least 8 records and 3 per class, 2..20 classes, unique ids. BYOD accepts a genes-as-columns CSV, a JSON array, or JSONL.
- **Species:** the pinned vocabulary is human. Mouse or other non-human identifiers will not resolve, and the validation stage says so rather than silently dropping them.
- **Privacy:** cells you bring are your responsibility. Do not upload confidential or restricted data — patient-derived or unpublished cells included — to a hosted runtime unless you are authorized to process it there. The default path uploads nothing and the BYOD branch keeps your file inside this runtime.
- **Expected log lines:** loading `BertForSequenceClassification` prints that the classifier head and pooler weights are newly initialised — that is the head this tutorial trains, not a defect.
- **External access:** the Hugging Face Hub only, to fetch the pinned `ctheodoris/Geneformer` snapshot (~421 MB in total) at revision `1f7fbae4e469…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `safetensors` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'geneformer-single-cell-pipeline',
    'repository_revision': 'uncommitted',
    'embedded_module': 'src/geneformer_single_cell_pipeline/pipeline.py',
    'embedded_modules': ['src/geneformer_single_cell_pipeline/metrics.py', 'src/geneformer_single_cell_pipeline/pipeline.py', 'src/geneformer_single_cell_pipeline/samples.py'],
    'module_sha256': 'e27aa254fe292820570496e3fef01bae1c4c568723ed84c3b3d349a6bf093b54',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, safetensors
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'safetensors': safetensors.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/geneformer_single_cell_pipeline/` @ `uncommitted`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/geneformer_single_cell_pipeline/metrics.py`

In [ ]:
"""Classification metrics and trivial baselines for Geneformer cell-state classification.

Pure Python (no scikit-learn): accuracy, macro-F1, per-class precision/recall/F1/support, and AUROC
(binary: positive class = the last entry of `classes`; multiclass: macro one-vs-rest), computed by
the Mann-Whitney rank statistic with average ranks for ties.
"""

from __future__ import annotations

from collections.abc import Mapping, Sequence
from typing import Any


def _prf(hits: int, n_pred: int, n_true: int) -> dict[str, float]:
    precision = hits / n_pred if n_pred else 0.0
    recall = hits / n_true if n_true else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4)}


def auroc(y_true: Sequence[int], scores: Sequence[float]) -> float | None:
    """Area under the ROC curve for binary 0/1 labels; None when only one class is present."""
    n_pos = sum(1 for y in y_true if y == 1)
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    order = sorted(range(len(scores)), key=lambda i: scores[i])
    ranks = [0.0] * len(scores)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and scores[order[j + 1]] == scores[order[i]]:
            j += 1
        avg = (i + j + 2) / 2.0  # 1-based average rank of the tie block
        for k in range(i, j + 1):
            ranks[order[k]] = avg
        i = j + 1
    rank_sum = sum(r for r, y in zip(ranks, y_true, strict=True) if y == 1)
    return round((rank_sum - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg), 4)


def classification_metrics(
    y_true: Sequence[str],
    y_pred: Sequence[str],
    scores: Sequence[Sequence[float]] | None,
    classes: Sequence[str],
) -> dict[str, Any]:
    """Discrete and ranking metrics over one evaluation split (labels are class names).

    `scores[i][k]` is the score of class `classes[k]` for record i (softmax outputs from the
    pipeline; any monotone score works for AUROC). Class order is preserved exactly as given.
    """
    if len(y_true) != len(y_pred):
        raise ValueError(f"{len(y_true)} labels vs {len(y_pred)} predictions")
    class_list = list(classes)
    unknown = sorted((set(y_true) | set(y_pred)) - set(class_list))
    if unknown:
        raise ValueError(f"labels outside the class list {class_list}: {unknown}")
    n = len(y_true)
    correct = sum(1 for t, p in zip(y_true, y_pred, strict=True) if t == p)
    per_class: dict[str, dict[str, Any]] = {}
    f1s: list[float] = []
    for c in class_list:
        hits = sum(1 for t, p in zip(y_true, y_pred, strict=True) if t == c and p == c)
        n_pred = sum(1 for p in y_pred if p == c)
        n_true = sum(1 for t in y_true if t == c)
        prf = _prf(hits, n_pred, n_true)
        per_class[c] = {**prf, "support": n_true, "predicted": n_pred}
        if n_true:
            f1s.append(prf["f1"])
    result: dict[str, Any] = {
        "n": n,
        "accuracy": round(correct / n, 4) if n else 0.0,
        "macro_f1": round(sum(f1s) / len(f1s), 4) if f1s else 0.0,
        "per_class": per_class,
        "classes": class_list,
        "decision_rule": "argmax over class scores",
        "auroc": None,
        "auroc_definition": None,
    }
    if scores is not None and n:
        if len(scores) != n or any(len(row) != len(class_list) for row in scores):
            raise ValueError("scores must be one row per record with one column per class")
        if len(class_list) == 2:
            pos = class_list[-1]
            result["auroc"] = auroc([1 if t == pos else 0 for t in y_true], [row[-1] for row in scores])
            result["auroc_definition"] = f"binary AUROC with positive class {pos!r} (last class in the list)"
        else:
            values = []
            for k, c in enumerate(class_list):
                a = auroc([1 if t == c else 0 for t in y_true], [row[k] for row in scores])
                if a is not None:
                    values.append(a)
            result["auroc"] = round(sum(values) / len(values), 4) if values else None
            result["auroc_definition"] = "macro-averaged one-vs-rest AUROC over classes present in the split"
    return result


def majority_baseline(
    train_records: Sequence[Mapping[str, Any]],
    eval_records: Sequence[Mapping[str, Any]],
    classes: Sequence[str],
) -> dict[str, Any]:
    """Predict the most frequent training class for every evaluation record (EVAL11)."""
    counts: dict[str, int] = {}
    for r in train_records:
        counts[r["label"]] = counts.get(r["label"], 0) + 1
    majority = max(sorted(counts), key=counts.__getitem__)
    metrics = classification_metrics(
        [r["label"] for r in eval_records], [majority] * len(eval_records), None, classes
    )
    return {"baseline": "majority-class", "predicted_label": majority, **metrics}


def library_size_baseline(
    train_records: Sequence[Mapping[str, Any]],
    eval_records: Sequence[Mapping[str, Any]],
    classes: Sequence[str],
) -> dict[str, Any]:
    """Threshold on a cell's total counts, fitted on the training split only (binary tasks).

    Library size is the first thing that separates cells in a badly designed single-cell experiment,
    so it is the baseline worth ruling out. The threshold and the class direction are chosen to
    maximise training accuracy; the evaluation split is never touched during fitting (SPL8). On the
    tutorial sample every cell carries the same total counts by construction, so this baseline is
    expected to sit at chance -- which is the point: it shows the model is reading rank order.
    """
    class_list = list(classes)
    if len(class_list) != 2:
        raise ValueError("library_size_baseline is defined for binary tasks only")
    lo, hi = class_list

    def total(record: Mapping[str, Any]) -> float:
        return float(sum(record["counts"].values()))

    train_x = [total(r) for r in train_records]
    train_y = [r["label"] for r in train_records]
    best = (-1.0, 0.0, True)  # accuracy, threshold, high_is_hi
    for t in sorted(set(train_x)):
        for high_is_hi in (True, False):
            pred = [(hi if (x >= t) == high_is_hi else lo) for x in train_x]
            acc = sum(p == y for p, y in zip(pred, train_y, strict=True)) / len(train_y)
            if acc > best[0]:
                best = (acc, t, high_is_hi)
    _, threshold, high_is_hi = best
    eval_x = [total(r) for r in eval_records]
    eval_pred = [(hi if (x >= threshold) == high_is_hi else lo) for x in eval_x]
    span = max(eval_x) - min(eval_x) or 1.0
    normalised = [(x - min(eval_x)) / span for x in eval_x]
    scores = [[1.0 - x, x] if high_is_hi else [x, 1.0 - x] for x in normalised]
    metrics = classification_metrics([r["label"] for r in eval_records], eval_pred, scores, class_list)
    return {
        "baseline": "library-size threshold",
        "threshold": round(threshold, 2),
        "rule": f"predict {hi!r} when total counts {'>=' if high_is_hi else '<'} {threshold:.0f}",
        "train_accuracy": round(best[0], 4),
        **metrics,
    }

**Module 2/3:** `src/geneformer_single_cell_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""Geneformer V2-104M (`ctheodoris/Geneformer`) DIMER pipeline: verified snapshot, rank-value
encoding of single-cell transcriptomes, cell embeddings, and bounded cell-state classification
fine-tuning with a portable adapter artifact.

Everything model-related is imported lazily so that snapshot verification and input validation run
(and can refuse) before `torch` or `transformers` are imported (fleet RTM-001).
"""

from __future__ import annotations

import hashlib
import json
import pickle
import warnings
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

MODEL_ID = "ctheodoris/Geneformer"
MODEL_REVISION = "1f7fbae4e469a5f4f1af8c111a529cfe1b3829f5"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "geneformer-v2-104m"
# The repository hosts several checkpoints; this pipeline packages exactly one of them.
MODEL_SUBFOLDER = "Geneformer-V2-104M"
DICTIONARY_DIR = "geneformer"
TOKEN_DICTIONARY_FILE = "geneformer/token_dictionary_gc104M.pkl"
GENE_MEDIAN_DICTIONARY_FILE = "geneformer/gene_median_dictionary_gc104M.pkl"
GENE_NAME_DICTIONARY_FILE = "geneformer/gene_name_id_dict_gc104M.pkl"
ARTIFACT_FORMAT = "org.valcorza.geneformer-single-cell.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Ceilings. The V2 checkpoint's config.json declares max_position_embeddings 4096 and the upstream
# README states an input size of 4096 tokens for V2; `<cls>` and `<eos>` take two of them, so 4094
# ranked genes is the most a cell can carry. Longer rank lists are truncated at the tail (the
# lowest-ranked genes), which is the upstream behaviour, and the pipeline reports the truncation.
MAX_INPUT_TOKENS = 4096
MAX_GENES_PER_CELL = MAX_INPUT_TOKENS - 2
MAX_CELLS_PER_CALL = 32
HIDDEN_SIZE = 768  # config.json hidden_size; the width of every `embed` row
VOCAB_SIZE = 20275  # config.json vocab_size; equals len(token_dictionary)
SPECIAL_TOKENS = ("<pad>", "<mask>", "<cls>", "<eos>")
MIN_DETECTED_GENES = 10  # a cell with fewer measured genes cannot produce a meaningful ranking

# The only globals the pinned data pickles legitimately contain: the gene-median dictionary stores
# numpy float64 scalars. Everything else is refused, so loading these files parses data rather than
# executing arbitrary code (see docs/WEIGHTS.md, "Dictionary trust boundary").
ALLOWED_PICKLE_GLOBALS = frozenset(
    {
        # numpy 1.x and numpy 2.x spell the scalar reconstructor differently; both appear in the
        # wild for the same pinned bytes depending on the numpy that reads them.
        ("numpy.core.multiarray", "scalar"),
        ("numpy._core.multiarray", "scalar"),
        ("numpy", "dtype"),
    }
)


def _is_allowed_global(module: str, name: str) -> bool:
    """The allowlist plus numpy 2's concrete dtype classes (`numpy.dtypes.Float64DType`)."""
    if (module, name) in ALLOWED_PICKLE_GLOBALS:
        return True
    return module == "numpy.dtypes" and name.endswith("DType")


class RestrictedUnpickler(pickle.Unpickler):
    """Unpickler that refuses every global except the allowlist above."""

    def find_class(self, module: str, name: str) -> Any:  # noqa: D102
        if not _is_allowed_global(module, name):
            raise pickle.UnpicklingError(
                f"refusing to resolve {module}.{name} while loading a pinned data dictionary; "
                f"only {sorted(ALLOWED_PICKLE_GLOBALS)} are allowed"
            )
        return super().find_class(module, name)


def load_data_pickle(path: str | Path) -> dict[Any, Any]:
    """Load one digest-verified data dictionary through `RestrictedUnpickler`."""
    with open(path, "rb") as fh:
        obj = RestrictedUnpickler(fh).load()
    if not isinstance(obj, dict):
        raise ValueError(f"{path}: expected a dict, got {type(obj).__name__}")
    return obj


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = hashlib.sha256()
        with open(file_path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                digest.update(chunk)
        if digest.hexdigest() != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest.hexdigest()} != manifest {entry['sha256']}")
    return manifest


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the Geneformer snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    return _verify_manifest(root, MODEL_ID, MODEL_REVISION)


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at the pinned revision straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest entries that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


@dataclass(frozen=True)
class GeneVocabulary:
    """The pinned gene dictionaries: Ensembl id -> token, Ensembl id -> corpus median, symbol -> id."""

    tokens: dict[str, int]
    medians: dict[str, float]
    symbol_to_id: dict[str, str]

    @property
    def special(self) -> dict[str, int]:
        return {name: self.tokens[name] for name in SPECIAL_TOKENS if name in self.tokens}

    @property
    def gene_ids(self) -> list[str]:
        """Ensembl ids that have both a token and a corpus median (the encodable vocabulary)."""
        return sorted(gid for gid in self.tokens if gid in self.medians)

    def resolve(self, name: str) -> str | None:
        """Map a gene symbol to its Ensembl id; an Ensembl id passes through unchanged."""
        if name in self.tokens:
            return name
        return self.symbol_to_id.get(name)


def load_gene_vocabulary(path: str | Path | None = None) -> GeneVocabulary:
    """Load the three pinned dictionaries through the restricted unpickler and sanity-check them."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    tokens = {str(k): int(v) for k, v in load_data_pickle(root / TOKEN_DICTIONARY_FILE).items()}
    medians = {str(k): float(v) for k, v in load_data_pickle(root / GENE_MEDIAN_DICTIONARY_FILE).items()}
    symbols = {str(k): str(v) for k, v in load_data_pickle(root / GENE_NAME_DICTIONARY_FILE).items()}
    if len(tokens) != VOCAB_SIZE:
        raise ValueError(f"token dictionary holds {len(tokens)} entries, config declares {VOCAB_SIZE}")
    missing_special = [name for name in SPECIAL_TOKENS if name not in tokens]
    if missing_special:
        raise ValueError(f"token dictionary is missing special tokens {missing_special}")
    return GeneVocabulary(tokens, medians, symbols)


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "1..MAX_CELLS_PER_CALL cells, each a mapping of Ensembl gene id (or gene symbol) "
        "to a non-negative count"
    ),
    "cells": [1, MAX_CELLS_PER_CALL],
    "detected_genes_per_cell": [MIN_DETECTED_GENES, MAX_GENES_PER_CELL],
    "input_tokens": MAX_INPUT_TOKENS,
    "preprocessing": (
        "counts are normalised to a fixed library size, divided by the gene's corpus median from the "
        "pinned gene_median dictionary, ranked in descending order, truncated to the input size, and "
        "mapped to gene tokens between <cls> and <eos> (Geneformer rank-value encoding)"
    ),
}


def _check_cells(cells: Any, names: Any = None) -> tuple[list[dict[str, float]], list[str]]:
    """Raise TypeError/ValueError naming the first violated rule; return (cells, ids).

    ``embed``, ``classify`` and ``validate_inputs`` all route through this function so their
    acceptance criteria cannot diverge.
    """
    if isinstance(cells, Mapping) or not isinstance(cells, Sequence) or isinstance(cells, str | bytes):
        raise TypeError("cells must be a list of {gene: count} mappings (one mapping per cell)")
    if not 1 <= len(cells) <= MAX_CELLS_PER_CALL:
        raise ValueError(f"cells must hold 1..{MAX_CELLS_PER_CALL} items, got {len(cells)}")
    checked: list[dict[str, float]] = []
    for i, cell in enumerate(cells):
        if not isinstance(cell, Mapping):
            raise TypeError(f"cells[{i}] must be a mapping of gene to count, got {type(cell).__name__}")
        if not cell:
            raise ValueError(f"cells[{i}] is empty")
        counts: dict[str, float] = {}
        for gene, value in cell.items():
            if not isinstance(gene, str) or not gene.strip():
                raise TypeError(f"cells[{i}] has a non-string gene key {gene!r}")
            if isinstance(value, bool) or not isinstance(value, int | float):
                raise TypeError(f"cells[{i}][{gene!r}] must be a number, got {type(value).__name__}")
            if value < 0 or value != value or value in (float("inf"), float("-inf")):
                raise ValueError(f"cells[{i}][{gene!r}] must be a finite non-negative count, got {value!r}")
            counts[gene] = float(value)
        detected = sum(1 for v in counts.values() if v > 0)
        if detected < MIN_DETECTED_GENES:
            raise ValueError(
                f"cells[{i}] has {detected} detected gene(s); at least {MIN_DETECTED_GENES} are required "
                "to rank a transcriptome"
            )
        checked.append(counts)
    if names is None:
        ids = [f"cell-{i}" for i in range(len(checked))]
    else:
        if isinstance(names, str | bytes) or not isinstance(names, Sequence) or len(names) != len(checked):
            raise ValueError("names must be a list with exactly one id per cell")
        ids = [str(n) for n in names]
        if len(set(ids)) != len(ids):
            raise ValueError("names must be unique")
    return checked, ids


def rank_value_encode(
    counts: Mapping[str, float],
    vocabulary: GeneVocabulary,
    *,
    target_sum: float = 10_000.0,
    max_genes: int = MAX_GENES_PER_CELL,
) -> dict[str, Any]:
    """Geneformer rank-value encoding of one cell; returns the token ids and what was dropped.

    Genes are normalised to `target_sum` total counts, divided by their corpus median expression,
    ranked in descending order and truncated to `max_genes`. Genes with a zero count, genes absent
    from the token dictionary, and genes without a corpus median are dropped and reported rather
    than silently ignored (VAL7).
    """
    unknown: list[str] = []
    no_median: list[str] = []
    resolved: dict[str, float] = {}
    for gene, value in counts.items():
        if value <= 0:
            continue
        gene_id = vocabulary.resolve(gene)
        if gene_id is None:
            unknown.append(gene)
            continue
        if gene_id not in vocabulary.medians:
            no_median.append(gene_id)
            continue
        resolved[gene_id] = resolved.get(gene_id, 0.0) + value
    if not resolved:
        raise ValueError(
            "no gene in this cell could be encoded: none of the detected genes carry both a Geneformer "
            "token and a corpus median (check that gene identifiers are human Ensembl ids or symbols)"
        )
    total = sum(resolved.values())
    scaled = {gid: (value / total) * target_sum / vocabulary.medians[gid] for gid, value in resolved.items()}
    ranked = sorted(scaled.items(), key=lambda item: (-item[1], item[0]))
    kept = ranked[:max_genes]
    return {
        "tokens": [vocabulary.tokens[gid] for gid, _ in kept],
        "ranked_gene_ids": [gid for gid, _ in kept],
        "n_detected": sum(1 for v in counts.values() if v > 0),
        "n_encoded": len(resolved),
        "n_kept": len(kept),
        "n_truncated": max(0, len(ranked) - len(kept)),
        "unknown_genes": sorted(set(unknown)),
        "genes_without_median": sorted(set(no_median)),
        "library_size": total,
    }


def validate_inputs(
    cells: Sequence[Mapping[str, float]],
    vocabulary: GeneVocabulary,
    *,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: encode every cell and return the input manifest (schema, observations, verdict).

    Rejection is reported by raising exactly as ``embed``/``classify`` would.
    """
    checked, ids = _check_cells(cells, names)
    rows = []
    for cid, counts in zip(ids, checked, strict=True):
        encoded = rank_value_encode(counts, vocabulary)
        rows.append(
            {
                "id": cid,
                "detected_genes": encoded["n_detected"],
                "encoded_genes": encoded["n_encoded"],
                "tokens_kept": encoded["n_kept"],
                "genes_truncated": encoded["n_truncated"],
                "unknown_genes": len(encoded["unknown_genes"]),
                "genes_without_median": len(encoded["genes_without_median"]),
                "library_size": encoded["library_size"],
            }
        )
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": rows,
        "n_cells": len(checked),
        "max_tokens_observed": max(row["tokens_kept"] for row in rows),
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "model_subfolder": MODEL_SUBFOLDER,
    }


def _softmax(logits: Sequence[float]) -> list[float]:
    import math

    top = max(logits)
    exps = [math.exp(v - top) for v in logits]
    total = sum(exps)
    return [v / total for v in exps]


@dataclass
class GeneformerPipeline:
    """Geneformer V2-104M pipeline: `embed` always; `classify` after `adapt` or `from_artifact`."""

    _embedder: Callable[[list[list[int]]], list[list[float]]]
    device: str
    vocabulary: GeneVocabulary
    load_warnings: list[str] = field(default_factory=list)
    classes: list[str] = field(default_factory=list)
    _classifier: Callable[[list[list[int]]], list[list[float]]] | None = None
    model: Any = None
    classifier_model: Any = None
    weights_dir: Path | None = None
    adaptation: dict[str, Any] = field(default_factory=dict)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> GeneformerPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if not (root / MANIFEST_NAME).is_file():
            raise FileNotFoundError(f"no snapshot manifest at {root} and allow_download={allow_download}")
        # Stage, verify and load the data dictionaries before importing model libraries (RTM-001).
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        vocabulary = load_gene_vocabulary(root)
        import torch
        from transformers import BertModel

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model = BertModel.from_pretrained(
                str(root / MODEL_SUBFOLDER),
                local_files_only=True,
                trust_remote_code=False,
                use_safetensors=True,
                add_pooling_layer=False,
            )
        model = model.to(resolved_device).eval()
        messages = [f"{w.category.__name__}: {w.message}" for w in caught]
        pipe = cls(
            cls._make_embedder(model, resolved_device, vocabulary),
            resolved_device,
            vocabulary,
            messages,
        )
        pipe.model, pipe.weights_dir = model, root
        return pipe

    # -- backends ---------------------------------------------------------------------------------

    @staticmethod
    def _batch(token_rows: list[list[int]], vocabulary: GeneVocabulary, device: str) -> dict[str, Any]:
        """Pad rank-value rows into `<cls> … <eos>` batches with an attention mask."""
        import torch

        cls_id, eos_id, pad_id = (vocabulary.tokens[t] for t in ("<cls>", "<eos>", "<pad>"))
        rows = [[cls_id, *row, eos_id] for row in token_rows]
        width = max(len(row) for row in rows)
        input_ids = [row + [pad_id] * (width - len(row)) for row in rows]
        attention = [[1] * len(row) + [0] * (width - len(row)) for row in rows]
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long, device=device),
            "attention_mask": torch.tensor(attention, dtype=torch.long, device=device),
        }

    @classmethod
    def _make_embedder(
        cls, model: Any, device: str, vocabulary: GeneVocabulary
    ) -> Callable[[list[list[int]]], list[list[float]]]:
        import torch

        def embedder(token_rows: list[list[int]]) -> list[list[float]]:
            batch = cls._batch(token_rows, vocabulary, device)
            # no_grad, not inference_mode: tensors produced here must stay usable by a later
            # training epoch that shares this module.
            with torch.no_grad():
                hidden = model(**batch).last_hidden_state
            mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
            return pooled.float().cpu().tolist()

        return embedder

    @classmethod
    def _make_classifier(
        cls, model: Any, device: str, vocabulary: GeneVocabulary
    ) -> Callable[[list[list[int]]], list[list[float]]]:
        import torch

        def classifier(token_rows: list[list[int]]) -> list[list[float]]:
            batch = cls._batch(token_rows, vocabulary, device)
            with torch.no_grad():
                logits = model(**batch).logits
            return logits.float().cpu().tolist()

        return classifier

    def encode(self, cells: Sequence[Mapping[str, float]]) -> list[dict[str, Any]]:
        """Rank-value encode every cell with this pipeline's pinned vocabulary."""
        return [rank_value_encode(cell, self.vocabulary) for cell in cells]

    # -- public stages ----------------------------------------------------------------------------

    def embed(
        self, cells: Sequence[Mapping[str, float]], *, names: Sequence[str] | None = None
    ) -> dict[str, Any]:
        """Mean-pooled last-hidden-state cell embedding (HIDDEN_SIZE floats per cell)."""
        checked, ids = _check_cells(cells, names)
        encoded = self.encode(checked)
        vectors = self._embedder([e["tokens"] for e in encoded])
        if len(vectors) != len(checked) or any(len(v) != HIDDEN_SIZE for v in vectors):
            raise RuntimeError("backend returned embeddings of the wrong shape")
        return {
            "ids": ids,
            "embeddings": [[float(x) for x in v] for v in vectors],
            "dimension": HIDDEN_SIZE,
            "pooling": "mean of the last hidden state over the cell's tokens (padding excluded)",
            "unit": "one vector per cell; representations, not predictions",
            "tokens_kept": [e["n_kept"] for e in encoded],
            "genes_truncated": [e["n_truncated"] for e in encoded],
            "n_cells": len(checked),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def classify(
        self, cells: Sequence[Mapping[str, float]], *, names: Sequence[str] | None = None
    ) -> dict[str, Any]:
        """Cell-state scores and argmax label; requires a prior `adapt` or `from_artifact`."""
        if self._classifier is None or not self.classes:
            raise RuntimeError(
                "classify requires an adapted head: call adapt(...) or load from_artifact(...) first"
            )
        checked, ids = _check_cells(cells, names)
        encoded = self.encode(checked)
        logits = self._classifier([e["tokens"] for e in encoded])
        predictions = []
        for cid, enc, row in zip(ids, encoded, logits, strict=True):
            if len(row) != len(self.classes):
                raise RuntimeError("backend returned a logits row that does not match the class list")
            scores = _softmax(row)
            best = max(range(len(scores)), key=scores.__getitem__)
            predictions.append(
                {
                    "id": cid,
                    "tokens_kept": enc["n_kept"],
                    "label": self.classes[best],
                    "score": scores[best],
                    "scores": dict(zip(self.classes, scores, strict=True)),
                }
            )
        return {
            "predictions": predictions,
            "classes": list(self.classes),
            "decision_rule": (
                "argmax over softmax(logits); scores are softmax outputs, not calibrated probabilities"
            ),
            "n_cells": len(checked),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "adaptation": dict(self.adaptation),
        }

    def evaluate(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Held-out cell-state classification metrics (see metrics.classification_metrics)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import classification_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        validate_dataset(records, vocabulary=self.vocabulary, classes=self.classes)
        predicted: list[str] = []
        scores: list[list[float]] = []
        for start in range(0, len(records), MAX_CELLS_PER_CALL):
            chunk = records[start : start + MAX_CELLS_PER_CALL]
            result = self.classify([r["counts"] for r in chunk], names=[r["id"] for r in chunk])
            for p in result["predictions"]:
                predicted.append(p["label"])
                scores.append([p["scores"][c] for c in self.classes])
        return classification_metrics([r["label"] for r in records], predicted, scores, self.classes)

    def adapt(
        self,
        train_records: Sequence[Mapping[str, Any]],
        val_records: Sequence[Mapping[str, Any]] | None = None,
        *,
        classes: Sequence[str] | None = None,
        epochs: int = 4,
        learning_rate: float = 2e-4,
        batch_size: int = 4,
        trainable_layers: int = 2,
        weight_decay: float = 0.01,
        seed: int = 42,
    ) -> dict[str, Any]:
        """Bounded gradient fine-tuning of a cell-state classification head on the verified base.

        Builds `BertForSequenceClassification` from the pinned checkpoint (the head is newly
        initialised), freezes every parameter except the head and the last `trainable_layers`
        encoder layers, and runs AdamW for `epochs` passes. Validation records are monitored per
        epoch only; the final epoch's weights are kept (no selection).
        """
        if self.model is None or self.weights_dir is None:
            raise RuntimeError("adapt requires a pipeline built by from_pretrained (no loaded base model)")
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if not 1 <= int(epochs) <= 50:
            raise ValueError("epochs must be in 1..50 (tutorial-scale adaptation)")
        if not 1 <= int(batch_size) <= MAX_CELLS_PER_CALL:
            raise ValueError(f"batch_size must be in 1..{MAX_CELLS_PER_CALL}")
        if not 0 <= int(trainable_layers) <= 12:
            raise ValueError("trainable_layers must be in 0..12 (the checkpoint has 12 encoder layers)")
        train_manifest = validate_dataset(train_records, vocabulary=self.vocabulary, classes=classes)
        class_list = list(train_manifest["classes"])
        if val_records is not None:
            validate_dataset(val_records, vocabulary=self.vocabulary, classes=class_list)

        import random

        import torch
        from transformers import BertForSequenceClassification

        random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        clf = BertForSequenceClassification.from_pretrained(
            str(self.weights_dir / MODEL_SUBFOLDER),
            local_files_only=True,
            trust_remote_code=False,
            use_safetensors=True,
            num_labels=len(class_list),
        ).to(self.device)
        for p in clf.parameters():
            p.requires_grad = False
        layers = clf.bert.encoder.layer
        for layer in layers[len(layers) - int(trainable_layers) :] if trainable_layers else []:
            for p in layer.parameters():
                p.requires_grad = True
        for module in (clf.classifier, clf.bert.pooler):
            if module is not None:
                for p in module.parameters():
                    p.requires_grad = True
        trainable = [n for n, p in clf.named_parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in clf.parameters() if p.requires_grad)
        n_total = sum(p.numel() for p in clf.parameters())
        optimizer = torch.optim.AdamW(
            [p for p in clf.parameters() if p.requires_grad], lr=learning_rate, weight_decay=weight_decay
        )
        label_index = {c: i for i, c in enumerate(class_list)}
        encoded = [
            (e["tokens"], label_index[r["label"]])
            for r, e in zip(train_records, self.encode([r["counts"] for r in train_records]), strict=True)
        ]
        self.classes = class_list
        self.classifier_model = clf
        self._classifier = self._make_classifier(clf, self.device, self.vocabulary)

        history: list[dict[str, Any]] = []
        for epoch in range(1, int(epochs) + 1):
            clf.train()
            order = list(range(len(encoded)))
            random.shuffle(order)
            total_loss, n_batches = 0.0, 0
            for start in range(0, len(order), int(batch_size)):
                rows = [encoded[i] for i in order[start : start + int(batch_size)]]
                batch = self._batch([tokens for tokens, _ in rows], self.vocabulary, self.device)
                labels = torch.tensor([y for _, y in rows], device=self.device)
                optimizer.zero_grad()
                out = clf(**batch, labels=labels)
                out.loss.backward()
                optimizer.step()
                total_loss += float(out.loss.item())
                n_batches += 1
            clf.eval()
            entry: dict[str, Any] = {
                "epoch": epoch,
                "train_loss": round(total_loss / max(1, n_batches), 6),
                "n_batches": n_batches,
            }
            if val_records:
                val = self.evaluate(val_records)
                entry["val_accuracy"] = val["accuracy"]
                entry["val_macro_f1"] = val["macro_f1"]
            history.append(entry)
        clf.eval()
        self.adaptation = {
            "method": "gradient fine-tuning (AdamW) of the classification head and pooler"
            + (f" and the last {int(trainable_layers)} encoder layer(s)" if trainable_layers else ""),
            "classes": class_list,
            "epochs": int(epochs),
            "learning_rate": float(learning_rate),
            "batch_size": int(batch_size),
            "weight_decay": float(weight_decay),
            "trainable_layers": int(trainable_layers),
            "seed": int(seed),
            "precision": "float32",
            "trainable_parameters": int(n_trainable),
            "total_parameters": int(n_total),
            "trainable_parameter_names": trainable,
            "train_records": len(train_records),
            "val_records": len(val_records) if val_records else 0,
            "selection": "final epoch kept; validation metrics are monitoring only",
            "history": history,
        }
        return dict(self.adaptation)

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Export the trainable tensors as safetensors plus a JSON manifest binding them to the base."""
        if self.classifier_model is None or not self.classes:
            raise RuntimeError("save_artifact requires an adapted head (call adapt first)")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adaptation.get("trainable_parameter_names", []))
        tensors = {
            k: v.detach().cpu().contiguous()
            for k, v in self.classifier_model.state_dict().items()
            if k in names
        }
        if not tensors:
            raise RuntimeError("no trainable tensors recorded; nothing to export")
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path))
        digest = hashlib.sha256(weights_path.read_bytes()).hexdigest()
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
                "subfolder": MODEL_SUBFOLDER,
                "license": MODEL_LICENSE,
            },
            "classes": list(self.classes),
            "files": [
                {"path": ARTIFACT_WEIGHTS_NAME, "bytes": weights_path.stat().st_size, "sha256": digest}
            ],
            "tensors": sorted(tensors),
            "adaptation": {k: v for k, v in self.adaptation.items() if k != "trainable_parameter_names"},
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Rebuild the classification head from an exported artifact (manifest verified before loading)."""
        if self.model is None or self.weights_dir is None:
            raise RuntimeError("load_artifact requires a pipeline built by from_pretrained")
        art = Path(artifact_dir)
        manifest_path = art / ARTIFACT_MANIFEST_NAME
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest not found: {manifest_path}")
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        base = manifest.get("base_model", {})
        if (base.get("model_id"), base.get("model_revision"), base.get("subfolder")) != (
            MODEL_ID,
            MODEL_REVISION,
            MODEL_SUBFOLDER,
        ):
            raise ValueError(
                f"artifact was trained on {base}, this package pins "
                f"{MODEL_ID}@{MODEL_REVISION} ({MODEL_SUBFOLDER})"
            )
        classes = [str(c) for c in manifest.get("classes", [])]
        if len(classes) < 2 or len(set(classes)) != len(classes):
            raise ValueError("artifact manifest must list at least two unique classes")
        for entry in manifest["files"]:
            fp = art / entry["path"]
            if not fp.is_file():
                raise FileNotFoundError(f"artifact file missing: {fp}")
            if fp.stat().st_size != entry["bytes"]:
                raise ValueError(f"{entry['path']}: size {fp.stat().st_size} != manifest {entry['bytes']}")
            if hashlib.sha256(fp.read_bytes()).hexdigest() != entry["sha256"]:
                raise ValueError(f"{entry['path']}: sha256 mismatch against the artifact manifest")
        from safetensors.torch import load_file
        from transformers import BertForSequenceClassification

        clf = BertForSequenceClassification.from_pretrained(
            str(self.weights_dir / MODEL_SUBFOLDER),
            local_files_only=True,
            trust_remote_code=False,
            use_safetensors=True,
            num_labels=len(classes),
        )
        tensors = load_file(str(art / ARTIFACT_WEIGHTS_NAME))
        if set(tensors) != set(manifest.get("tensors", [])):
            raise ValueError("artifact tensors do not match the names listed in its manifest")
        _missing, unexpected = clf.load_state_dict(tensors, strict=False)
        if unexpected:
            raise ValueError(
                f"artifact carries tensors the base architecture does not have: {sorted(unexpected)[:5]}"
            )
        clf = clf.to(self.device).eval()
        self.classes = classes
        self.classifier_model = clf
        self._classifier = self._make_classifier(clf, self.device, self.vocabulary)
        self.adaptation = {**manifest.get("adaptation", {}), "loaded_from_artifact": str(art)}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> GeneformerPipeline:
        """Verified base snapshot + exported adapter, ready for `classify`."""
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe

**Module 3/3:** `src/geneformer_single_cell_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Deterministic in-code sample cells and the labelled-dataset contract for cell-state classification.

The tutorial dataset is synthetic but uses **real human Ensembl gene ids** drawn from the pinned
Geneformer vocabulary, and it is built so that raw library size carries no signal: every cell is
scaled to the same total counts (integer rounding leaves a spread of about 0.1 %), and the two
classes are distinguished only by *which* of two gene programmes sits above its corpus median --
that is, by the rank order the model reads, not by how much RNA was captured. This is sanity
evidence for the adaptation contract, not biology (NOTEBOOK_SPEC 2.0 DAT8).
"""

from __future__ import annotations

import csv
import hashlib
import json
import random
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_GENES_PER_CELL, MIN_DETECTED_GENES, GeneVocabulary` removed — names are kernel globals defined by the carried modules

DATASET_REPRESENTATION = "io.github.kurtvalcorza.dataset.single-cell.rank-value-labels.v1"
SAMPLE_CLASSES: tuple[str, ...] = ("programme-A", "programme-B")
SAMPLE_SEED = 20260918
SAMPLE_SIZE = 64  # 32 per class
PROGRAMME_GENES = 24  # per programme; the two programmes are median-matched pairs
BACKGROUND_GENES = 300
LIBRARY_SIZE = 20_000  # every cell is scaled to this total, so library size cannot separate the classes
HIGH_FACTOR = 4.0  # programme genes above their corpus median in the class that expresses them
LOW_FACTOR = 0.25  # the same genes below their corpus median in the other class
MIN_RECORDS = 8
MAX_RECORDS = 2_000
MAX_CLASSES = 20
MIN_RECORDS_PER_CLASS = 3
MAX_ID_CHARS = 64
MAX_LABEL_CHARS = 64
REQUIRED_COLUMNS = ("id", "counts", "label")


def select_sample_genes(vocabulary: GeneVocabulary, seed: int = SAMPLE_SEED) -> dict[str, list[str]]:
    """Pick median-matched programme genes plus a background set, deterministically.

    Genes are drawn from the pinned vocabulary (Ensembl ids that carry both a token and a corpus
    median). The two programmes are built as pairs of genes with near-identical corpus medians, so
    that swapping which programme is high leaves every cell's raw count profile equally plausible.
    """
    rng = random.Random(seed)
    candidates = [gid for gid in vocabulary.gene_ids if 0.5 <= vocabulary.medians[gid] <= 5.0]
    candidates.sort(key=lambda gid: (vocabulary.medians[gid], gid))
    if len(candidates) < 2 * PROGRAMME_GENES + BACKGROUND_GENES:
        raise RuntimeError("the pinned vocabulary does not hold enough genes in the sampled median band")
    # Walk the median-sorted list in adjacent pairs so each A gene has a B partner of similar median.
    stride = len(candidates) // (PROGRAMME_GENES + 1)
    programme_a: list[str] = []
    programme_b: list[str] = []
    for k in range(PROGRAMME_GENES):
        index = (k + 1) * stride
        programme_a.append(candidates[index])
        programme_b.append(candidates[index + 1])
    chosen = set(programme_a) | set(programme_b)
    pool = [gid for gid in candidates if gid not in chosen]
    background = sorted(rng.sample(pool, BACKGROUND_GENES))
    return {"programme_a": programme_a, "programme_b": programme_b, "background": background}


def generate_sample_dataset(
    vocabulary: GeneVocabulary, seed: int = SAMPLE_SEED, size: int = SAMPLE_SIZE
) -> list[dict[str, Any]]:
    """`size` labelled cells (half `programme-A`, half `programme-B`), deterministic for a seed.

    Every cell is scaled to `LIBRARY_SIZE` total counts before rounding to integers. Background
    genes are drawn around their corpus median; the expressing programme's genes are placed above
    it and the other programme's below it, so the classes differ in rank order rather than in
    library size or gene detection.
    """
    if size < 2 or size % 2:
        raise ValueError("size must be an even number >= 2 (one cell per class per pair)")
    genes = select_sample_genes(vocabulary, seed)
    rng = random.Random(seed + 1)
    records: list[dict[str, Any]] = []
    for i in range(size // 2):
        for label in SAMPLE_CLASSES:
            high = genes["programme_a"] if label == "programme-A" else genes["programme_b"]
            low = genes["programme_b"] if label == "programme-A" else genes["programme_a"]
            weights: dict[str, float] = {}
            for gid in genes["background"]:
                weights[gid] = vocabulary.medians[gid] * rng.uniform(0.7, 1.3)
            for gid in high:
                weights[gid] = vocabulary.medians[gid] * HIGH_FACTOR * rng.uniform(0.9, 1.1)
            for gid in low:
                weights[gid] = vocabulary.medians[gid] * LOW_FACTOR * rng.uniform(0.9, 1.1)
            scale = LIBRARY_SIZE / sum(weights.values())
            counts = {gid: max(1, round(value * scale)) for gid, value in weights.items()}
            records.append(
                {
                    "id": f"{'a' if label == 'programme-A' else 'b'}-cell-{i:03d}",
                    "counts": counts,
                    "label": label,
                }
            )
    return records


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    """SHA-256 over the canonical (id, sorted counts, label) rows; recorded in provenance (OUT9)."""
    canon = json.dumps(
        [[r["id"], sorted(r["counts"].items()), r["label"]] for r in records], separators=(",", ":")
    )
    return hashlib.sha256(canon.encode("utf-8")).hexdigest()


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    vocabulary: GeneVocabulary | None = None,
    classes: Sequence[str] | None = None,
    min_records: int = MIN_RECORDS,
    min_per_class: int = MIN_RECORDS_PER_CLASS,
) -> dict[str, Any]:
    """Check a labelled cell dataset against the contract; return its manifest.

    Every error names the record and the violated rule (VAL4/DAT19). When `vocabulary` is supplied
    the manifest also reports how many genes per cell are encodable, which is the number that
    decides whether a cell can be ranked at all.
    """
    if isinstance(records, str | bytes | Mapping) or not isinstance(records, Sequence):
        raise TypeError("records must be a list of {'id', 'counts', 'label'} mappings")
    if len(records) < min_records:
        raise ValueError(f"dataset has {len(records)} records; at least {min_records} are required")
    if len(records) > MAX_RECORDS:
        raise ValueError(f"dataset has {len(records)} records; ceiling is {MAX_RECORDS}")
    seen_ids: set[str] = set()
    counts_by_class: dict[str, int] = {}
    detected: list[int] = []
    encodable: list[int] = []
    libraries: list[float] = []
    for i, rec in enumerate(records):
        if not isinstance(rec, Mapping):
            raise TypeError(f"record[{i}] must be a mapping, got {type(rec).__name__}")
        missing = [c for c in REQUIRED_COLUMNS if c not in rec]
        if missing:
            raise ValueError(
                f"record[{i}] is missing required column(s) {missing}; required: {list(REQUIRED_COLUMNS)}"
            )
        rid = str(rec["id"]).strip()
        if not rid or len(rid) > MAX_ID_CHARS:
            raise ValueError(f"record[{i}] id must be 1..{MAX_ID_CHARS} characters")
        if rid in seen_ids:
            raise ValueError(f"record[{i}] duplicates id {rid!r}")
        seen_ids.add(rid)
        cell = rec["counts"]
        if not isinstance(cell, Mapping) or not cell:
            raise ValueError(f"record[{i}] ({rid}) counts must be a non-empty mapping of gene to count")
        n_detected = 0
        total = 0.0
        for gene, value in cell.items():
            if not isinstance(gene, str) or not gene.strip():
                raise TypeError(f"record[{i}] ({rid}) has a non-string gene key {gene!r}")
            if isinstance(value, bool) or not isinstance(value, int | float):
                raise TypeError(f"record[{i}] ({rid}) count for {gene!r} must be a number")
            if value < 0 or value != value or value in (float("inf"), float("-inf")):
                raise ValueError(f"record[{i}] ({rid}) count for {gene!r} must be finite and non-negative")
            if value > 0:
                n_detected += 1
                total += float(value)
        if n_detected < MIN_DETECTED_GENES:
            raise ValueError(
                f"record[{i}] ({rid}) has {n_detected} detected gene(s); at least {MIN_DETECTED_GENES} "
                "are required to rank a transcriptome"
            )
        detected.append(n_detected)
        libraries.append(total)
        if vocabulary is not None:
            n_encodable = sum(
                1
                for gene, value in cell.items()
                if value > 0
                and (resolved := vocabulary.resolve(gene)) is not None
                and resolved in vocabulary.medians
            )
            if n_encodable < MIN_DETECTED_GENES:
                raise ValueError(
                    f"record[{i}] ({rid}) has {n_encodable} gene(s) that carry both a Geneformer token "
                    f"and a corpus median; at least {MIN_DETECTED_GENES} are required (are these human "
                    "Ensembl ids or gene symbols?)"
                )
            encodable.append(n_encodable)
        label = rec["label"]
        if not isinstance(label, str) or not label.strip() or len(label) > MAX_LABEL_CHARS:
            raise ValueError(
                f"record[{i}] ({rid}) label must be a non-empty string of at most {MAX_LABEL_CHARS} chars"
            )
        counts_by_class[label] = counts_by_class.get(label, 0) + 1
    if classes is None:
        class_list = sorted(counts_by_class)
    else:
        class_list = [str(c) for c in classes]
        unknown = sorted(set(counts_by_class) - set(class_list))
        if unknown:
            raise ValueError(f"labels {unknown} are not in the class list {class_list}")
    if len(class_list) < 2:
        raise ValueError(f"classification needs at least 2 classes, found {class_list}")
    if len(class_list) > MAX_CLASSES:
        raise ValueError(f"{len(class_list)} classes exceeds the ceiling of {MAX_CLASSES}")
    thin = [c for c in class_list if counts_by_class.get(c, 0) < min_per_class]
    if thin:
        raise ValueError(f"classes {thin} have fewer than {min_per_class} records each (class coverage rule)")
    manifest: dict[str, Any] = {
        "verdict": "accepted",
        "representation": DATASET_REPRESENTATION,
        "n_records": len(records),
        "classes": class_list,
        "class_counts": {c: counts_by_class.get(c, 0) for c in class_list},
        "detected_genes": {
            "min": min(detected),
            "max": max(detected),
            "mean": round(sum(detected) / len(detected), 1),
        },
        "library_size": {
            "min": min(libraries),
            "max": max(libraries),
            "mean": round(sum(libraries) / len(libraries), 1),
        },
        "ceilings": {
            "max_genes_per_cell": MAX_GENES_PER_CELL,
            "min_detected_genes": MIN_DETECTED_GENES,
            "max_records": MAX_RECORDS,
            "max_classes": MAX_CLASSES,
            "min_records": min_records,
            "min_records_per_class": min_per_class,
        },
        "digest": dataset_digest(records),
        "findings": [],
    }
    if encodable:
        manifest["encodable_genes"] = {
            "min": min(encodable),
            "max": max(encodable),
            "mean": round(sum(encodable) / len(encodable), 1),
        }
    return manifest


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.2,
    test_fraction: float = 0.25,
    seed: int = 42,
) -> dict[str, list[dict[str, Any]]]:
    """Stratified random train/validation/test split (assumes independent cells, SPL3).

    Real single-cell data is rarely independent — cells from one donor, plate or batch belong
    together — so a deployment must split by that grouping instead. The tutorial's cells are
    generated independently, which is why a random split is honest here.
    """
    if not (0.0 < val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("val_fraction and test_fraction must be in (0, 1) and sum to less than 1")
    manifest = validate_dataset(records)
    rng = random.Random(seed)
    by_class: dict[str, list[dict[str, Any]]] = {c: [] for c in manifest["classes"]}
    for rec in records:
        by_class[rec["label"]].append(dict(rec))
    out: dict[str, list[dict[str, Any]]] = {"train": [], "validation": [], "test": []}
    for cls in manifest["classes"]:
        rows = by_class[cls]
        rng.shuffle(rows)
        n_val = max(1, round(len(rows) * val_fraction))
        n_test = max(1, round(len(rows) * test_fraction))
        if len(rows) - n_val - n_test < 1:
            raise ValueError(f"class {cls!r} has {len(rows)} records; too few to leave one per split")
        out["validation"].extend(rows[:n_val])
        out["test"].extend(rows[n_val : n_val + n_test])
        out["train"].extend(rows[n_val + n_test :])
    for part in out.values():
        rng.shuffle(part)
    return out


def load_byod_dataset(source: str | Path) -> list[dict[str, Any]]:
    """Read a user-supplied cell dataset (JSON array, JSONL, or a genes-as-columns CSV).

    CSV shape: first column `id`, last column `label`, every other column a gene id or symbol whose
    cells hold counts; zero counts are dropped (they carry no rank). JSON/JSONL records are
    `{"id": ..., "counts": {gene: count, ...}, "label": ...}`. Nothing is renamed or rescaled
    (VAL7); the records are then validated with `validate_dataset`.
    """
    path = Path(source)
    if not path.is_file():
        raise FileNotFoundError(f"BYOD dataset file not found: {path}")
    text = path.read_text(encoding="utf-8-sig")
    if not text.strip():
        raise ValueError(f"BYOD dataset file is empty: {path}")
    suffix = path.suffix.lower()
    records: list[dict[str, Any]] = []
    if suffix == ".csv":
        reader = csv.reader(text.splitlines())
        header = [h.strip() for h in next(reader, [])]
        if len(header) < 3 or header[0] != "id" or header[-1] != "label":
            raise ValueError(
                "CSV must start with an 'id' column, end with a 'label' column, and carry one gene per "
                f"column in between; got header {header[:3]}...{header[-1:]}"
            )
        genes = header[1:-1]
        for line_no, row in enumerate(reader, start=2):
            if not row:
                continue
            if len(row) != len(header):
                raise ValueError(f"line {line_no} has {len(row)} fields, header has {len(header)}")
            counts: dict[str, float] = {}
            for gene, value in zip(genes, row[1:-1], strict=True):
                value = value.strip()
                if not value:
                    continue
                try:
                    number = float(value)
                except ValueError as exc:
                    raise ValueError(f"line {line_no}, gene {gene!r}: {value!r} is not a number") from exc
                if number > 0:
                    counts[gene] = number
            records.append({"id": row[0].strip(), "counts": counts, "label": row[-1].strip()})
    elif suffix == ".jsonl":
        for line_no, line in enumerate(text.splitlines(), start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"line {line_no} is not valid JSON: {exc}") from exc
    elif suffix == ".json":
        try:
            data = json.loads(text)
        except json.JSONDecodeError as exc:
            raise ValueError(f"file is not valid JSON: {exc}") from exc
        if not isinstance(data, list):
            raise TypeError("JSON dataset must be a top-level array of objects")
        records = data
    else:
        raise ValueError(f"unsupported BYOD file type {suffix!r}; use .csv, .json or .jsonl")
    validate_dataset(records)
    return [dict(r) for r in records]


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write records in the BYOD CSV shape (`id`, one column per gene, `label`) as a template."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    genes = sorted({gene for r in records for gene in r["counts"]})
    with open(out, "w", encoding="utf-8", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(["id", *genes, "label"])
        for r in records:
            writer.writerow([r["id"], *(r["counts"].get(gene, 0) for gene in genes), r["label"]])
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `6`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `1f7fbae4e469…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `GeneformerPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "geneformer-v2-104m",
  "modelId": "ctheodoris/Geneformer",
  "revision": "1f7fbae4e469a5f4f1af8c111a529cfe1b3829f5",
  "files": [
    {
      "path": "Geneformer-V2-104M/config.json",
      "bytes": 590,
      "sha256": "467d4492f0dd53b4d60afffe20812db484ca1cf9fdbeb6a6e060e93564f70859"
    },
    {
      "path": "Geneformer-V2-104M/model.safetensors",
      "bytes": 417571156,
      "sha256": "fff5cba29ddd8792991fa77b4872246fbe548a178cebda3775cdc72b67780e7f"
    },
    {
      "path": "README.md",
      "bytes": 9470,
      "sha256": "56d6e570b349cbedae9a54634421c94e7af8ea467ce0fdd79193372ae3cbdbd8"
    },
    {
      "path": "geneformer/gene_median_dictionary_gc104M.pkl",
      "bytes": 1512661,
      "sha256": "a51c53f6a771d64508dfaf61529df70e394c53bd20856926117ae5d641a24bf5"
    },
    {
      "path": "geneformer/gene_name_id_dict_gc104M.pkl",
      "bytes": 1660882,
      "sha256": "fabfa0c2f49c598c59ae432a32c3499a5908c033756c663b5e0cddf58deea8e1"
    },
    {
      "path": "geneformer/token_dictionary_gc104M.pkl",
      "bytes": 425590,
      "sha256": "67c445f4385127adfc48dcc072320cd65d6822829bf27dd38070e6e787bc597f"
    }
  ],
  "totalBytes": 421180349
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = GeneformerPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Gene dictionaries, sample cells, validation and split

Geneformer's tokenizer is not a vocabulary file: it is three dictionaries shipped with the model — gene id to token, gene id to corpus median expression, and gene symbol to gene id. They are Python pickles, so this pipeline loads them through `RestrictedUnpickler`, which refuses every global except the numpy scalar types the median dictionary legitimately contains; an unexpected class raises instead of executing (NOTEBOOK_SPEC 2.0 §20). They have already been digest-verified as part of the snapshot in Section 3.

The default dataset is then generated in code with a fixed seed: 32 cells per class over real human Ensembl ids, each scaled to the same library size. `validate_dataset` checks the schema, the count values, the per-cell gene coverage and the class coverage before any model runs, and — because it is given the vocabulary — also reports how many genes per cell can actually be encoded. `split_dataset` shuffles **within each class** and cuts 20 % validation / 25 % test.

Look for: 20,275 tokens and 42,005 medians loaded, 64 cells, classes `['programme-A', 'programme-B']`, splits 36/12/16, and a written `outputs/geneformer_single_cell_sample_dataset.csv` — the exact file shape BYOD expects. To use your own cells, set `USE_BYOD = True` and re-run from this cell.

In [ ]:
import json
import os
from pathlib import Path

USE_BYOD = False  # @param {type:"boolean"}
VAL_FRACTION = 0.2  # @param {type:"number"}
TEST_FRACTION = 0.25  # @param {type:"number"}
SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
vocabulary = pipe.vocabulary
print({'tokens': len(vocabulary.tokens), 'corpus_medians': len(vocabulary.medians), 'gene_symbols': len(vocabulary.symbol_to_id), 'encodable_gene_ids': len(vocabulary.gene_ids), 'special_tokens': vocabulary.special})

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    data_source = 'BYOD (' + file_name + ')'
else:
    records = generate_sample_dataset(vocabulary)
    data_source = f'synthetic median-matched programme dataset (seed {SAMPLE_SEED}, {SAMPLE_SIZE} cells)'

dataset_manifest = validate_dataset(records, vocabulary=vocabulary)
CLASSES = dataset_manifest['classes']
splits = split_dataset(records, val_fraction=VAL_FRACTION, test_fraction=TEST_FRACTION, seed=SEED)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
write_dataset_csv(records, 'outputs/geneformer_single_cell_sample_dataset.csv')

print({'data_source': data_source, 'n_records': dataset_manifest['n_records'], 'classes': CLASSES, 'class_counts': dataset_manifest['class_counts']})
print({'detected_genes': dataset_manifest['detected_genes'], 'encodable_genes': dataset_manifest.get('encodable_genes'), 'library_size': dataset_manifest['library_size']})
print({'ceilings': dataset_manifest['ceilings'], 'digest': dataset_manifest['digest'][:16] + '...'})
print({'train': len(train_records), 'validation': len(val_records), 'test': len(test_records)})

## 5. Read one rank-value encoding

This is the cell to slow down on. `rank_value_encode` takes one cell's counts, normalises them to a fixed library size, divides each gene by its corpus median, ranks the genes by that ratio and maps them to tokens. The printout compares the top of the ranking for one cell of each class: the cells detect the same genes with near-identical total counts, yet the ranked heads differ, because the two classes put different median-matched programmes above their corpus median.

The encoding also reports what it dropped — genes with a zero count, genes with no Geneformer token, genes with no corpus median — and how many ranked genes were truncated at the 4,094-token input limit. Nothing is dropped silently (VAL7).

In [ ]:
example_a = next(r for r in records if r['label'] == CLASSES[0])
example_b = next(r for r in records if r['label'] == CLASSES[1])
encoded_a = rank_value_encode(example_a['counts'], vocabulary)
encoded_b = rank_value_encode(example_b['counts'], vocabulary)

def summarise(record, encoded):
    return {
        'id': record['id'],
        'label': record['label'],
        'library_size': encoded['library_size'],
        'detected': encoded['n_detected'],
        'encoded': encoded['n_encoded'],
        'tokens_kept': encoded['n_kept'],
        'truncated': encoded['n_truncated'],
        'unknown_genes': len(encoded['unknown_genes']),
        'genes_without_median': len(encoded['genes_without_median']),
    }

print(summarise(example_a, encoded_a))
print(summarise(example_b, encoded_b))
print('top 8 ranked genes,', example_a['label'] + ':', encoded_a['ranked_gene_ids'][:8])
print('top 8 ranked genes,', example_b['label'] + ':', encoded_b['ranked_gene_ids'][:8])
print('same genes detected in both cells:', set(example_a['counts']) == set(example_b['counts']))
print('first 8 token ids:', encoded_a['tokens'][:8])

## 6. Cell embeddings (representation, not prediction)

`pipe.embed` runs the verified encoder over the rank-value tokens and returns one 768-dimensional vector per cell: the mean of the last hidden state over the cell's tokens, padding excluded. Embeddings are representations — they carry no label and no metric of their own; a downstream labelled task is what gives them meaning (EVAL9). The cell embeds eight validation cells, writes them with their ids to `outputs/geneformer_single_cell_embeddings.csv` (OUT4), and prints the mean cosine similarity within and between classes as an inspection, not an evaluation.

In [ ]:
import csv
import math

embed_records = val_records[:8]
embedding_result = pipe.embed([r['counts'] for r in embed_records], names=[r['id'] for r in embed_records])
vectors = embedding_result['embeddings']
print({'n_cells': embedding_result['n_cells'], 'dimension': embedding_result['dimension'], 'tokens_kept': embedding_result['tokens_kept'][:4], 'pooling': embedding_result['pooling']})

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

within, between = [], []
for i in range(len(embed_records)):
    for j in range(i + 1, len(embed_records)):
        sim = cosine(vectors[i], vectors[j])
        (within if embed_records[i]['label'] == embed_records[j]['label'] else between).append(sim)
print({'mean_cosine_within_class': round(sum(within) / len(within), 4) if within else None, 'mean_cosine_between_classes': round(sum(between) / len(between), 4) if between else None, 'note': 'inspection only; embeddings are unlabelled representations'})

with open('outputs/geneformer_single_cell_embeddings.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'label'] + [f'dim_{k}' for k in range(embedding_result['dimension'])])
    for r, vec in zip(embed_records, vectors):
        writer.writerow([r['id'], r['label']] + [f'{x:.6f}' for x in vec])
print('wrote outputs/geneformer_single_cell_embeddings.csv')

## 7. Baselines on the test split

Two trivial predictors set the floor before any training (EVAL10/EVAL11). `majority_baseline` predicts the most frequent training class for every test cell — 0.5 accuracy on a balanced split. `library_size_baseline` fits one threshold on total counts per cell using the training split only (SPL8); library size is the first confounder to rule out in any single-cell classification, and here it is uninformative by construction, so the baseline should land near chance. A fine-tuned model that clears both has learned something about the gene ranking rather than about sequencing depth.

In [ ]:
baseline_majority = majority_baseline(train_records, test_records, CLASSES)
print({k: baseline_majority[k] for k in ('baseline', 'predicted_label', 'accuracy', 'macro_f1')})
if len(CLASSES) == 2:
    baseline_library = library_size_baseline(train_records, test_records, CLASSES)
    print({k: baseline_library[k] for k in ('baseline', 'rule', 'train_accuracy', 'accuracy', 'macro_f1', 'auroc')})
else:
    baseline_library = None
    print('library-size baseline is defined for binary tasks only; skipped for', len(CLASSES), 'classes')

## 8. Bounded fine-tuning

`pipe.adapt` builds `BertForSequenceClassification` from the verified checkpoint (the head and pooler are newly initialised — the log line says so), freezes every parameter except the head, the pooler and the last `TRAINABLE_LAYERS` encoder layers, and runs AdamW with the hyperparameters below (FT4/FT6): tutorial values chosen for a few minutes of CPU, not production settings. Validation metrics are computed after each epoch for **monitoring only**; the final epoch's weights are kept, so no selection happens on the validation split (EVAL14). Training loss going down is optimisation evidence, not task-quality evidence (FT7) — Section 9 is where quality is measured.

Look for about 14.8 M trainable parameters of 104 M, and validation accuracy leaving 0.5 partway through training. If it is still at 0.5 in the last epoch the head is under-trained rather than broken: the AUROC in the next section will be high while accuracy sits at chance, which means the ranking is right and the decision boundary has not moved yet.

In [ ]:
import time

EPOCHS = 4  # @param {type:"integer"}
LEARNING_RATE = 2e-4  # @param {type:"number"}
BATCH_SIZE = 4  # @param {type:"integer"}
TRAINABLE_LAYERS = 2  # @param {type:"integer"}

started = time.perf_counter()
adapt_result = pipe.adapt(
    train_records,
    val_records,
    classes=CLASSES,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    trainable_layers=TRAINABLE_LAYERS,
    seed=SEED,
)
adapt_seconds = round(time.perf_counter() - started, 1)
print({'method': adapt_result['method'], 'trainable_parameters': adapt_result['trainable_parameters'], 'total_parameters': adapt_result['total_parameters'], 'precision': adapt_result['precision'], 'device': pipe.device, 'seconds': adapt_seconds})
for step in adapt_result['history']:
    print(step)

## 9. Held-out evaluation

`pipe.evaluate` classifies every cell of a split and reports `accuracy` (discrete correctness under the argmax rule), `macro_f1` (the unweighted mean of per-class F1, which exposes a model that ignores a class), per-class precision/recall/F1 with support, and `auroc` (ranking quality of the positive-class score, independent of the argmax threshold). The **test split** was never used for training or monitoring, so its numbers are the independent evidence (SPL6/SPL7). These are tutorial metrics on a synthetic 16-cell split (EVAL6): one holdout, no dispersion estimate. The report, with both baselines and the deltas against them, is written to `outputs/geneformer_single_cell_evaluation_report.json`.

In [ ]:
val_metrics = pipe.evaluate(val_records)
test_metrics = pipe.evaluate(test_records)
print({'split': 'validation', **{k: val_metrics[k] for k in ('n', 'accuracy', 'macro_f1', 'auroc')}})
print({'split': 'test', **{k: test_metrics[k] for k in ('n', 'accuracy', 'macro_f1', 'auroc')}})
for cls_name, row in test_metrics['per_class'].items():
    print({'class': cls_name, **row})

evaluation_report = {
    'task': 'single-cell state classification (bounded fine-tuning of Geneformer V2-104M)',
    'evidence': 'tutorial sample-sanity metrics on one stratified holdout; not a benchmark and not biology',
    'estimation': 'single train/validation/test split, seed ' + str(SEED) + ', no dispersion estimate',
    'data_source': data_source,
    'dataset_digest': dataset_manifest['digest'],
    'classes': CLASSES,
    'splits': {'train': len(train_records), 'validation': len(val_records), 'test': len(test_records)},
    'baselines': {'majority': baseline_majority, 'library_size': baseline_library},
    'validation_metrics': val_metrics,
    'test_metrics': test_metrics,
    'delta_vs_majority': {k: round(test_metrics[k] - baseline_majority[k], 4) for k in ('accuracy', 'macro_f1')},
    'adaptation': {k: v for k, v in adapt_result.items() if k != 'trainable_parameter_names'},
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/geneformer_single_cell_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report, f, indent=2)
print({'delta_vs_majority': evaluation_report['delta_vs_majority'], 'report': 'outputs/geneformer_single_cell_evaluation_report.json'})

## 10. Inference on new cells

`pipe.classify` returns, per cell, the argmax `label`, its `score` and the full `scores` dictionary in class order. The scores are softmax outputs of a head trained on a few dozen cells — **not calibrated probabilities** (UNC2); the only decision rule is argmax (UNC3), and a deployment that must trade false positives against false negatives owns its own threshold. On the default path the new cells are generated with a different seed, so their true labels are known and shown as a check; on the BYOD path the first six test-split cells stand in as new data (INF2). Predictions are written to `outputs/geneformer_single_cell_predictions.csv` with ids and per-class scores.

In [ ]:
if USE_BYOD:
    new_records = test_records[:6]
    new_source = 'first six BYOD test-split cells'
else:
    new_records = generate_sample_dataset(vocabulary, seed=7, size=6)
    new_source = 'freshly generated cells (seed 7)'
input_manifest = validate_inputs([r['counts'] for r in new_records], vocabulary, names=[r['id'] for r in new_records])
print({'new_source': new_source, 'verdict': input_manifest['verdict'], 'n_cells': input_manifest['n_cells'], 'max_tokens_observed': input_manifest['max_tokens_observed']})
inference_result = pipe.classify([r['counts'] for r in new_records], names=[r['id'] for r in new_records])
predictions = inference_result['predictions']
print({'decision_rule': inference_result['decision_rule']})
n_match = 0
for p, r in zip(predictions, new_records):
    n_match += p['label'] == r['label']
    print({'id': p['id'], 'predicted': p['label'], 'score': round(p['score'], 4), 'true_label': r['label']})
print({'matches': n_match, 'of': len(new_records), 'note': 'sanity check on generated labels, not an evaluation'})

with open('outputs/geneformer_single_cell_predictions.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'tokens_kept', 'predicted_label', 'score'] + [f'score_{c}' for c in CLASSES])
    for p in predictions:
        writer.writerow([p['id'], p['tokens_kept'], p['label'], f"{p['score']:.6f}"] + [f"{p['scores'][c]:.6f}" for c in CLASSES])
print('wrote outputs/geneformer_single_cell_predictions.csv')

## 11. Export the adapter and verify a fresh reload

`pipe.save_artifact` writes only the trained tensors (head, pooler and the unfrozen encoder layers) as `adapter.safetensors` plus a `manifest.json` that records the artifact format, the exact base model id, revision **and subfolder** the tensors belong to (ART4 — the repository hosts several checkpoints, so the subfolder is part of the identity), the class order, the tensor names, the file size and SHA-256, and the adaptation configuration (OUT8). `GeneformerPipeline.from_artifact` then re-verifies the base snapshot, checks the artifact manifest and digests **before** deserialising, rebuilds the classifier and overlays the tensors — a fresh object from files, not the in-memory model (VER2). The cell compares its predictions on the same new cells with the pre-export ones: labels must match exactly and scores within `1e-5` (VER4).

In [ ]:
artifact_dir = Path('outputs/geneformer_single_cell_adapter')
pipe.save_artifact(artifact_dir, metadata={'data_source': data_source, 'dataset_digest': dataset_manifest['digest'], 'test_metrics': {k: test_metrics[k] for k in ('n', 'accuracy', 'macro_f1', 'auroc')}})
with open(artifact_dir / ARTIFACT_MANIFEST_NAME, encoding='utf-8') as f:
    artifact_manifest = json.load(f)
print({'format': artifact_manifest['format'], 'base_model': artifact_manifest['base_model'], 'classes': artifact_manifest['classes'], 'n_tensors': len(artifact_manifest['tensors']), 'files': artifact_manifest['files']})

reloaded_pipe = GeneformerPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR)
reloaded_result = reloaded_pipe.classify([r['counts'] for r in new_records], names=[r['id'] for r in new_records])
max_score_diff = 0.0
for before, after in zip(predictions, reloaded_result['predictions']):
    assert before['id'] == after['id'] and before['label'] == after['label'], f'reload parity failure on {before["id"]}'
    max_score_diff = max(max_score_diff, abs(before['score'] - after['score']))
assert max_score_diff < 1e-5, f'reload score drift {max_score_diff}'
print({'reload_parity': 'PASS', 'labels_equal': True, 'max_abs_score_diff': max_score_diff, 'loaded_from': reloaded_pipe.adaptation.get('loaded_from_artifact')})

## 12. Result export and provenance

The last output, `outputs/geneformer_single_cell_result.json`, gathers everything a reader needs to interpret the files above: the notebook source revision, the model id, immutable revision, subfolder and licence, the dataset source and digest, the adaptation configuration, baseline and held-out metrics, the new-cell predictions, the artifact manifest, the reload-parity result, and the runtime versions and device (OUT6/OUT7). No credential is involved anywhere in this notebook, so none can leak into it (OUT10).

In [ ]:
import platform

result_payload = {
    'task': 'single-cell state classification adaptation (Geneformer V2-104M)',
    'pipeline_class': 'GeneformerPipeline',
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_subfolder': MODEL_SUBFOLDER,
    'model_license': MODEL_LICENSE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'notebook_source': NOTEBOOK_SOURCE,
    'data_source': data_source,
    'dataset_manifest': dataset_manifest,
    'vocabulary': {'tokens': len(vocabulary.tokens), 'corpus_medians': len(vocabulary.medians), 'special_tokens': vocabulary.special},
    'encoding_example': {'id': example_a['id'], 'label': example_a['label'], 'top_ranked_gene_ids': encoded_a['ranked_gene_ids'][:8], 'tokens_kept': encoded_a['n_kept'], 'truncated': encoded_a['n_truncated']},
    'embedding_summary': {'n_cells': embedding_result['n_cells'], 'dimension': embedding_result['dimension'], 'pooling': embedding_result['pooling']},
    'evaluation_report': evaluation_report,
    'inference': {'new_source': new_source, 'decision_rule': inference_result['decision_rule'], 'predictions': predictions},
    'artifact_format': ARTIFACT_FORMAT,
    'artifact_format_version': ARTIFACT_FORMAT_VERSION,
    'artifact_manifest': artifact_manifest,
    'reload_parity': {'labels_equal': True, 'max_abs_score_diff': max_score_diff},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'safetensors': safetensors.__version__,
        'device': pipe.device,
        'precision': 'float32',
    },
}
with open('outputs/geneformer_single_cell_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_payload, f, indent=2)

print('outputs/:')
for path in sorted(Path('outputs').rglob('*')):
    if path.is_file():
        print(f'  - {path.as_posix()} ({path.stat().st_size / 1024:.1f} KB)')

## Interpretation and limits

The fine-tuned head separates two classes of cells that share their library size, detect the same genes and differ only in which median-matched gene programme ranks higher — which the library-size baseline cannot do. That is the whole claim of this notebook: the rank-value encoding carries the signal, and a bounded adaptation can pick it up. The test split has 16 synthetic cells, the metrics come from one seeded holdout with no dispersion estimate, and the classes are defined by a generator rule rather than by biology — so a perfect score here says the adaptation contract works, not that Geneformer predicts any real cell state.

On real data the same workflow needs more care than this sample shows. Cells from one donor, plate, or 10x run are not independent, so a random split leaks and a donor- or batch-level split is required. Labels transferred from a reference atlas carry that atlas's errors. Ambient RNA, doublets and dying cells change the ranking before the model sees it, and none of that is detected here. The pinned vocabulary is human; other species will not resolve. The softmax scores are uncalibrated, and the embeddings are representations that need a labelled downstream task before any quality can be stated.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model and its gene dictionaries, load those dictionaries without executing arbitrary code, rank-value encode cells, execute bounded fine-tuning, evaluate against trivial baselines on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, production fitness, or biological validity of the classes.

**Optional experiments (do not affect the default path):** set `TRAINABLE_LAYERS = 0` to train the head alone and compare the test metrics; lower `EPOCHS` to 2 to see the under-trained regime where AUROC is high but accuracy sits at 0.5 — ranking before thresholding; or bring a real labelled dataset through BYOD and read the library-size baseline first, because if it already scores well your labels may be predictable from sequencing depth alone.

## References

- Repository README: https://github.com/kurtvalcorza/geneformer-single-cell-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/geneformer-single-cell-pipeline/blob/main/MODEL_CARD.md
- Upstream model: https://huggingface.co/ctheodoris/Geneformer
- Theodoris, C. V. et al. (2023). Transfer learning enables predictions in network biology. *Nature* 618, 616–624. https://www.nature.com/articles/s41586-023-06139-9
- Chen, H. et al. (2024). Quantized multi-task learning for context-specific representations of gene network dynamics. bioRxiv 2024.08.16.608180. https://doi.org/10.1101/2024.08.16.608180